# claudeknows learning loop: learn from a walk

You film a "walk" around the house on your phone and put it in Google Drive under
`claudeknows_photos/walks/`. This notebook then, in one **Runtime -> Run all**:

1. turns new walk videos into photos (about 2 per second, blurry and repeated ones dropped);
2. lets two independent "labellers" (the current detector and OWLv2) look at every photo; only
   boxes they **both** agree on, and that stay steady over neighbouring photos, are used;
3. retrains **only if** the walks added enough new boxes (at least 30), and replaces the current
   model **only if** the new one is clearly better on your checked "gold" photos;
4. writes a plain-language summary to Drive (`claudeknows_models/loop_summary.txt`);
5. shows a few **yes/no questions** at the very end about photos the labellers were unsure of.

**About the questions:** "Run all" does not wait for you. Each click is saved to Drive straight
away and is used the **next** time you run this notebook. You don't have to answer them all.

**Before you start:** Runtime -> Change runtime type -> GPU (e.g. T4). You need a model from the
spatial notebook first (`claudeknows_models/detector.pt` in Drive). Nothing here is uploaded to GitHub.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"  # change to a claude/... branch to try unmerged work
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!git fetch -q origin $BRANCH && git reset -q --hard origin/$BRANCH  # always the latest code
!git log -1 --format='Code version: %h %s'
!pip install -q -r requirements.txt transformers

## 2. Connect Google Drive

Everything the loop keeps lives in Drive: your walk videos (`claudeknows_photos/walks/`), the walk
photos and your answers (`claudeknows_models/walk_frames/`), every model version, the gold set
(`claudeknows_models/gold/`) and the summaries. Colab's own disk is wiped when the session ends.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Could not connect Google Drive:", e)
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("Google Drive is NOT connected. Run this cell again, click 'Connect to Google "
                     "Drive' and tick every box. The loop needs Drive to keep your answers and models.")
PHOTOS = "/content/drive/MyDrive/claudeknows_photos"
MODELS = "/content/drive/MyDrive/claudeknows_models"
FRAMES = f"{MODELS}/walk_frames"
GOLD = f"{MODELS}/gold"
CURRENT = f"{MODELS}/detector.pt"
os.makedirs(FRAMES, exist_ok=True)
if not os.path.isfile(CURRENT):
    raise SystemExit("No current model in Drive (claudeknows_models/detector.pt). Run the spatial "
                     "notebook once first; it saves the model there.")
walks = sorted(os.listdir(f"{PHOTOS}/walks")) if os.path.isdir(f"{PHOTOS}/walks") else []
print(f"{len(walks)} file(s) in claudeknows_photos/walks/")
HAVE_GOLD = os.path.isfile(f"{GOLD}/gold.json")
print("Gold set:", "found" if HAVE_GOLD else "none yet, so no new model can replace the current one automatically")

## 3. Settings

- `EPOCHS`: training rounds when a retrain happens (same as the spatial notebook).
- `MEASURE_NOISE_FLOOR`: the first retrain with a gold set also trains the recipe with 5 seeds, to
  learn how much scores move by chance. That is 5 extra trainings, **once ever**; each is saved as
  soon as it finishes, so after a disconnect only the missing ones run next time.
- `MEANT_TO_COVER`: objects every walk should show, e.g. `("keys",)`. A retrain then also needs
  10 new boxes of each. Empty = only the "30 new boxes in total" rule.

In [ ]:
EPOCHS = 15
MEASURE_NOISE_FLOOR = True
MEANT_TO_COVER = ()

## 4. Run the loop

Import walks -> auto-label -> retrain + gold check (only if there is enough new data) -> summary.
Photos you answered last time are included now. A retrain takes about as long as the spatial
notebook's training (it also re-downloads the internet photos, since Colab's disk is new each time).

In [ ]:
import time
from claudeknows import loop
t0 = time.time()
label_fn = loop.colab_label_fn(CURRENT)
retrain_fn = loop.colab_retrain_fn("data/detection", MODELS, FRAMES, photos_dir=PHOTOS,
                                   gold_dir=GOLD if HAVE_GOLD else None, epochs=EPOCHS,
                                   noise_floor=MEASURE_NOISE_FLOOR)
out = loop.run_loop(f"{PHOTOS}/walks", MODELS, label_fn=label_fn, retrain_fn=retrain_fn,
                    frames_dir=FRAMES, covered=MEANT_TO_COVER)
print(f"(loop took {(time.time() - t0) / 60:.1f} minutes; summary saved to {MODELS}/loop_summary.txt)")

## 5. Your questions

Each photo shows one box. Click **Yes** if the box is around that object, **No** if it isn't.
Each click is saved to Drive at once and counts next time. Skip any you're unsure of.

In [ ]:
import io
import ipywidgets as W
from IPython.display import display

def question_widget(q):
    buf = io.BytesIO()
    loop.question_view(q["walk_dir"], q).save(buf, "JPEG")
    yes, no, saved = W.Button(description="Yes"), W.Button(description="No"), W.Label("")
    def click(answer):
        loop.answer(FRAMES, q["id"], answer, log=lambda *_: None)  # saved to Drive now
        yes.disabled = no.disabled = True
        saved.value = "saved: " + ("yes" if answer else "no")
    yes.on_click(lambda _b: click(True))
    no.on_click(lambda _b: click(False))
    return W.VBox([W.Image(value=buf.getvalue(), format="jpeg"),
                   W.Label(loop.question_text(q)), W.HBox([yes, no, saved])])

questions = loop.open_questions(FRAMES)
print(f"{len(questions)} question(s)" if questions else "No questions this time.")
for q in questions:
    display(question_widget(q))